# Trend Analytics Agent — interactive demo

A walkthrough of the **Investigator + Writer** two-agent system in this repo.

By the end of this notebook you will have:

1. Verified your Azure OpenAI credentials work from this kernel.
2. Built (or re-used) a small SQLite warehouse from the **UCI Online Retail II** dataset.
3. Run sample SQL against the warehouse so you know what's in it.
4. Called each of the 7 bounded analysis tools directly, by hand.
5. Run the **Investigator** agent, watched it pick tools step by step.
6. Run the **Writer** agent on the Investigator's findings.
7. Verified that every number in the briefing came from a real tool call (the "no fabricated claims" guardrail).
8. Edited the brief and re-run with your own question.

Open every cell, read the comment above it, then run it. Most cells take < 10 seconds; a couple of agent cells take ~30–60 seconds.

---

## Prerequisites

You should already have:

- This repo cloned to your working directory.
- A Python 3.10+ environment with `pip install -e .` already run (see `AZURE_ML_SETUP.md`).
- `az login --use-device-code` completed in the same terminal that launched Jupyter, **or** an Azure ML compute instance with managed identity.
- A `.env` file with `OPENAI_ENDPOINT` and `CHAT_DEPLOYMENT` filled in (copy from `.env.example`).

If anything below fails, the cell explains what to do.


In [ ]:
# Make `src/` importable and load .env. Run me first.
import sys
from pathlib import Path

ROOT = Path.cwd()
# Allow running the notebook from project root or from a subdir.
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

print(f"project root: {ROOT}")
print(f"src on path:  {SRC}")


### Install the project into this kernel

If you're using the default Azure ML `jupyter_env` kernel (or any env where you haven't run `pip install -e .` from a terminal), this cell makes the kernel self-sufficient: it installs this project in editable mode, which pulls in `pandas`, `openpyxl`, `azure-identity`, `openai`, `httpx`, and `python-dotenv`.

Idempotent — re-running is a quick no-op once everything is in place.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)])
print("dependencies ready in this kernel")

## 1. Environment and credentials

The agents read configuration from environment variables. The `Config` dataclass shows what was actually picked up.

If `OPENAI_ENDPOINT` is empty: copy `.env.example` to `.env`, fill in the endpoint and chat deployment name from your Azure OpenAI resource, then restart this kernel.


In [ ]:
from config import get_config

cfg = get_config()
print(f"OPENAI_ENDPOINT  = {cfg.openai_endpoint}")
print(f"CHAT_DEPLOYMENT  = {cfg.chat_deployment}")
print(f"WAREHOUSE_DB     = {cfg.warehouse_db}")
print(f"RUNS_DIR         = {cfg.runs_dir}")
print(f"REFERENCE_DATE   = {cfg.reference_date or '(auto from warehouse)'}")


### Verify the Azure credential

`DefaultAzureCredential` walks a chain (environment vars → managed identity → `az login` cache → VS Code → ...). On an Azure ML compute instance the managed identity link usually wins. On a laptop you'll need `az login --use-device-code` first.

The cell below tries to acquire a token for the Cognitive Services scope. If it succeeds, the agent will succeed too.

In [ ]:
from azure.identity import DefaultAzureCredential

try:
    cred = DefaultAzureCredential()
    token = cred.get_token("https://cognitiveservices.azure.com/.default")
    print(f"OK — got a token, expires at epoch {token.expires_on}")
except Exception as e:
    print("FAILED to acquire a token. Most common fix: run `az login --use-device-code` in the terminal that launched this kernel, then restart the kernel.")
    print(f"\nError: {e}")


## 2. Build the warehouse

The `scripts/ingest.py` script downloads the UCI Online Retail II dataset (~45 MB, CC BY 4.0) and builds `data/warehouse.db` with one table (`sales`) and one view (`sales_v`). It's idempotent: re-running uses the cached `.xlsx`.

If `data/warehouse.db` already exists you can skip this cell. To force a rebuild, delete the file first.

In [ ]:
import subprocess

db_path = ROOT / cfg.warehouse_db
if db_path.exists():
    size_mb = db_path.stat().st_size / 1_000_000
    print(f"warehouse already present at {db_path} ({size_mb:.1f} MB) — skipping ingest")
else:
    print("building warehouse — this takes ~2 minutes the first time…")
    result = subprocess.run(
        [sys.executable, str(ROOT / "scripts" / "ingest.py")],
        capture_output=True, text=True, cwd=ROOT,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError("ingest failed — see output above")


## 3. Sample queries to verify the data

These are plain SQL against `sales_v` so you can see the shape of the data the agent will be reasoning about. The agent itself never writes raw SQL — it only fills parameters in the bounded tools — but you, the human, can.

In [ ]:
from db import warehouse

with warehouse(db_path) as wh:
    n_rows, _ = wh.run_query("SELECT COUNT(*) AS n FROM sales_v")
    date_range, _ = wh.run_query(
        "SELECT MIN(invoice_date) AS first_day, MAX(invoice_date) AS last_day FROM sales_v"
    )
    n_countries, _ = wh.run_query("SELECT COUNT(DISTINCT country) AS n FROM sales_v")
    n_products, _ = wh.run_query("SELECT COUNT(DISTINCT stock_code) AS n FROM sales_v")

print(f"rows:        {n_rows[0]['n']:>10,}")
print(f"date range:  {date_range[0]['first_day']} → {date_range[0]['last_day']}")
print(f"countries:   {n_countries[0]['n']:>10,}")
print(f"products:    {n_products[0]['n']:>10,}")


In [ ]:
# Top 10 countries by revenue (lifetime). Useful sanity check —
# you'll see the UK dominate, which colours how the agent will decompose later.
with warehouse(db_path) as wh:
    rows, _ = wh.run_query("""
        SELECT country, ROUND(SUM(quantity * price), 2) AS revenue
        FROM sales_v
        GROUP BY country
        ORDER BY revenue DESC
        LIMIT 10
    """)
for r in rows:
    print(f"  {r['country']:<25} {r['revenue']:>15,.2f}")


In [ ]:
# Monthly revenue across the whole history. Eyeballing this will tell
# you what the agent is comparing when it asks for current_month vs prior_month.
with warehouse(db_path) as wh:
    rows, _ = wh.run_query("""
        SELECT invoice_month, ROUND(SUM(quantity * price), 2) AS revenue
        FROM sales_v
        GROUP BY invoice_month
        ORDER BY invoice_month
    """)
for r in rows:
    print(f"  {r['invoice_month']}   {r['revenue']:>15,.2f}")


## 4. The seven tools the Investigator can call

The Investigator does not see the database directly. It sees seven function-call schemas. Each tool wraps one parameterised SQL query with a 5-second timeout and a 10k-row cap.

Listing the tools by name and one-line description.

In [ ]:
from tools import TOOL_SCHEMAS

for t in TOOL_SCHEMAS:
    fn = t["function"]
    desc = fn["description"].split(". ")[0]   # first sentence only
    print(f"- {fn['name']:<26} {desc}")


### Call a tool by hand — `metric_overview`

This is exactly what the Investigator does on its first turn: get current vs prior values for the headline metrics. We're calling the same Python method, with the same arguments shape, so you can see the structure the agent works with.

In [ ]:
import json
from db import warehouse
from tools import AnalysisTools

with warehouse(db_path) as wh:
    ref_date = cfg.reference_date or wh.reference_date()
    print(f"reference_date = {ref_date} (the latest date in the warehouse)")

    tools = AnalysisTools(warehouse=wh, reference_date=ref_date)

    result = tools.metric_overview(
        metrics=["revenue", "units", "orders"],
        current_period={"name": "current_month"},
        comparison_period={"name": "prior_month"},
    )

print(json.dumps(result, indent=2, default=str))


### `dimension_decomposition`

Break revenue by country for the current month. The agent uses this to find where movement is concentrated.

In [ ]:
with warehouse(db_path) as wh:
    tools = AnalysisTools(warehouse=wh, reference_date=ref_date)
    result = tools.dimension_decomposition(
        metric="revenue",
        dimension="country",
        period={"name": "current_month"},
        top_n=5,
    )
print(json.dumps(result, indent=2, default=str))


### `top_contributors`

Rank countries by their **signed contribution** to the change in revenue, current month vs prior. Positive values added to revenue; negative values subtracted from it.

In [ ]:
with warehouse(db_path) as wh:
    tools = AnalysisTools(warehouse=wh, reference_date=ref_date)
    result = tools.top_contributors(
        metric="revenue",
        dimension="country",
        period_a={"name": "current_month"},
        period_b={"name": "prior_month"},
        top_n=5,
    )
print(json.dumps(result, indent=2, default=str))


## 5. Run the Investigator

Now we let the agent drive. We pass `verbose=True` so each tool call prints as it happens — that's the "watch the agent think" view.

This cell makes a real Azure OpenAI request and will take ~30–60 seconds depending on how many tool calls the model decides to make (capped at 15).

In [ ]:
from agents.investigator import Investigator

# We open the warehouse for the duration of the agent run, then close it.
with warehouse(db_path) as wh:
    ref_date = cfg.reference_date or wh.reference_date()
    investigator = Investigator(
        openai_endpoint=cfg.openai_endpoint,
        warehouse=wh,
        reference_date=ref_date,
        chat_deployment=cfg.chat_deployment,
        verbose=True,
    )
    print(f"reference_date = {ref_date}, deployment = {cfg.chat_deployment}\n")
    print("--- agent tool calls ---")
    result = investigator.investigate()
    print(f"\n--- done in {result.iterations} iteration(s), {len(result.tool_calls)} tool call(s) ---")


### Findings — what the Investigator concluded

The Investigator's output is structured JSON, not prose. Each finding cites the `tool_call_id`s that support it.

In [ ]:
print(json.dumps({"summary": result.summary, "findings": result.findings}, indent=2))


## 6. Run the Writer

The Writer has **no tools** and **no database access**. Its only inputs are:

- the Investigator's findings JSON,
- the full tool-call log.

That's the structural reason the Writer cannot fabricate numbers. There's no path for it to invent a value — it can only quote what's already in the log.

In [ ]:
from agents.writer import write_briefing
from IPython.display import Markdown, display

briefing_md = write_briefing(
    openai_endpoint=cfg.openai_endpoint,
    chat_deployment=cfg.chat_deployment,
    reference_date=ref_date,
    findings=result.findings,
    summary=result.summary,
    tool_calls=result.tool_calls,
)

display(Markdown(briefing_md))


## 7. Audit the briefing — "no fabricated claims" check

Success criterion 3 in the spec: zero hallucinated figures. We sketch the manual check here. Pull every number out of the briefing, then confirm each one appears in some tool result.

A real Validator agent would automate this; for the demo, eyeballing is enough.

In [ ]:
import re

# Crude number extractor: integers, decimals, percentages, with optional commas/sign.
NUMBER_RE = re.compile(r"-?[0-9][0-9,]*(?:\.[0-9]+)?%?")
numbers = sorted(set(NUMBER_RE.findall(briefing_md)))
print(f"numbers cited in briefing: {len(numbers)}")
print(numbers)

# Build a haystack of every result string the agent saw.
haystack = "\n".join(c["result"] for c in result.tool_calls)

unsupported = []
for n in numbers:
    bare = n.rstrip("%").replace(",", "")
    if bare not in haystack and n not in haystack:
        unsupported.append(n)

if unsupported:
    print(f"\n⚠️  {len(unsupported)} numbers not found verbatim in tool results:")
    for u in unsupported:
        print(f"  - {u}")
    print("\nNote: this is a *strict* check. The Writer is allowed to round (e.g. 12.345 → 12.3),")
    print("so a few mismatches may be legitimate rounding rather than fabrication. Inspect each.")
else:
    print("\n✅ every cited number appears verbatim in the tool log")


## 8. Try your own brief

The Investigator's behaviour is steered by the brief you pass to `investigate()`. Edit the string below and re-run to see how the tool path changes. Examples to try:

- `"Focus only on the United Kingdom — what changed?"`
- `"Compare this quarter to the same quarter last year."`
- `"Which products had the biggest declines this month?"`
- `"Is the daily revenue series trending up or down over the last 90 days?"`

Each one will produce a different chain of tool calls — that's the agentic part.

In [ ]:
my_brief = (
    "Focus on the United Kingdom this month vs last month. "
    "Identify the products that contributed most to the change."
)

with warehouse(db_path) as wh:
    investigator = Investigator(
        openai_endpoint=cfg.openai_endpoint,
        warehouse=wh,
        reference_date=ref_date,
        chat_deployment=cfg.chat_deployment,
        verbose=True,
    )
    print("--- agent tool calls ---")
    custom_result = investigator.investigate(brief=my_brief)
    print(f"\n--- {custom_result.iterations} iteration(s), {len(custom_result.tool_calls)} tool call(s) ---")

custom_briefing = write_briefing(
    openai_endpoint=cfg.openai_endpoint,
    chat_deployment=cfg.chat_deployment,
    reference_date=ref_date,
    findings=custom_result.findings,
    summary=custom_result.summary,
    tool_calls=custom_result.tool_calls,
)
display(Markdown(custom_briefing))


## 9. Where to go from here

- **Modify a tool.** Edit `src/tools.py` and add a new bounded SQL tool — e.g. `month_over_month_seasonality`. Add its schema to `TOOL_SCHEMAS`. The agent will discover and use it on the next run.
- **Tighten the system prompt.** Edit `SYSTEM_PROMPT` in `src/agents/investigator.py` to change significance thresholds or stopping rules.
- **Swap the dataset.** Replace `scripts/ingest.py` with one that loads your own CSV / SQL view as long as it produces a `sales_v` view with the same columns (`invoice_date`, `country`, `description`, `customer_id`, `quantity`, `price`).
- **Add a Validator agent.** A third agent (or just a deterministic Python pass) that re-checks every number in the briefing against the tool log, the way §7 above did manually.
- **Schedule it.** Once you trust the output, wire `python scripts/run.py` into a weekly cron / Azure ML pipeline.
